In [ ]:
!pip install duckdb

In [ ]:
import duckdb
import os
import glob
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
os.listdir('/content/drive/MyDrive/')

['IMG-20170509-WA0001.jpg',
 'IMG-20170506-WA0013.jpg',
 'DSC_0215.JPG',
 'DSC_0209.JPG',
 'IMG-20170504-WA0010.jpeg',
 'DSC_0218.JPG',
 'DSC_0226.JPG',
 'DSC_0220.JPG',
 'DSC_0225.JPG',
 'DSC_0208.JPG',
 'DSC_0207.JPG',
 'IMG-20170506-WA0018.jpg',
 'IMG-20170509-WA0007.jpg',
 'VID-20170507-WA0009.mp4',
 'DSC_0213.JPG',
 'IMG-20170509-WA0002.jpg',
 'IMG-20170506-WA0014.jpg',
 'DSC_0221.JPG',
 'IMG-20170506-WA0008.jpg',
 'IMG-20170506-WA0016.jpg',
 'DSC_0219.JPG',
 'DSC_0222.JPG',
 'DSC_0212.JPG',
 'IMG-20170509-WA0004.jpg',
 'IMG-20170506-WA0017.jpg',
 'DSC_0210.JPG',
 'DSC_0206.JPG',
 'DSC_0216.JPG',
 'IMG-20170502-WA0001.jpg',
 'DSC_0211.JPG',
 'IMG-20170506-WA0011.jpg',
 'DSC_0214.JPG',
 'DSC_0223.JPG',
 'VID-20170507-WA0010.mp4',
 'IMG-20170429-WA0007.jpg',
 'IMG-20170426-WA0004.jpg',
 'IMG-20170429-WA0009.jpg',
 'IMG-20170429-WA0008.jpg',
 'IMG-20170429-WA0006.jpg',
 'DSC_0199.JPG',
 'DSC_0200.JPG',
 'IMG-20170517-WA0001.jpg',
 'IMG-20170517-WA0005.jpg',
 'IMG-20170523-WA0013.jpg'

In [ ]:
ruta = '/content/drive/MyDrive/iansaura/sql_project'
os.listdir(ruta)

['Untitled0.ipynb',
 'logs_services.json',
 'logs_deployments.json',
 'logs_error_logs.json',
 'logs_access_logs.json',
 'logs_servers.json',
 'logs_incidents.json',
 'logs_alerts.json',
 'logs_performance_metrics.json',
 'logs_on_call_schedules.json',
 'logs_alert_rules.json']

In [ ]:
print(con.execute("SHOW TABLES").fetchdf())


                  name
0          access_logs
1          alert_rules
2               alerts
3          deployments
4           error_logs
5            incidents
6    on_call_schedules
7  performance_metrics
8              servers
9             services


In [ ]:
con = duckdb.connect()

ruta = '/content/drive/MyDrive/iansaura/sql_project'

for file in glob.glob(f"{ruta}/logs_*.json"):
    table_name = os.path.splitext(os.path.basename(file))[0].replace("logs_", "")

    con.execute(f"""
        CREATE OR REPLACE TABLE {table_name} AS
        SELECT * FROM read_json_auto('{file}')
    """)

    print(f"Tabla creada: {table_name}")

Tabla creada: services
Tabla creada: deployments
Tabla creada: error_logs
Tabla creada: access_logs
Tabla creada: servers
Tabla creada: incidents
Tabla creada: alerts
Tabla creada: performance_metrics
Tabla creada: on_call_schedules
Tabla creada: alert_rules


In [ ]:
con.execute("SHOW TABLES").fetchdf()

,name
0,access_logs
1,alert_rules
2,alerts
3,deployments
4,error_logs
5,incidents
6,on_call_schedules
7,performance_metrics
8,servers
9,services


In [ ]:
## 1. EXPLORACIÓN INICIAL
## ¿Cuántos registros? ¿Qué período cubren?

query = """
SELECT
    COUNT(*) as total_requests,
    MIN(timestamp) as primera_request,
    MAX(timestamp) as ultima_request,
    COUNT(DISTINCT user_id) as usuarios_unicos,
    COUNT(DISTINCT endpoint) as endpoints_unicos
FROM access_logs
"""

resultado = con.execute(query).fetchdf()
print(resultado)

   total_requests     primera_request      ultima_request  usuarios_unicos  \
0            5000 2024-01-01 08:55:34 2026-04-08 23:00:48             2823   

   endpoints_unicos  
0                11  


In [ ]:
## 2. ENDPOINTS MÁS USADOS
## ¿Qué endpoints reciben más tráfico?

query = """
SELECT
    endpoint,
    COUNT(*) as hits,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM access_logs), 2) as porcentaje
FROM access_logs
GROUP BY endpoint
ORDER BY hits DESC
LIMIT 10;
"""

resultado = con.execute(query).fetchdf()
print(resultado)

           endpoint  hits  porcentaje
0           /health   481        9.62
1       /api/orders   481        9.62
2          /metrics   480        9.60
3        /api/users   477        9.54
4     /api/payments   474        9.48
5  /api/auth/logout   473        9.46
6     /api/products   446        8.92
7   /api/auth/login   442        8.84
8       /api/search   439        8.78
9     /api/checkout   413        8.26


In [ ]:
## 3. ANÁLISIS DE ERRORES
## ¿Qué endpoints tienen más errores 500?

query = """
SELECT
    endpoint,
    COUNT(*) as total_errors,
    COUNT(DISTINCT user_id) as usuarios_afectados,
    ROUND(AVG(response_time_ms), 2) as avg_response_time
FROM access_logs
WHERE status_code >= 500
GROUP BY endpoint
ORDER BY total_errors DESC
LIMIT 10;
"""

resultado = con.execute(query).fetchdf()
print(resultado)

           endpoint  total_errors  usuarios_afectados  avg_response_time
0           /health           113                  82           16030.94
1   /api/auth/login           110                  73           14829.62
2       /api/orders           108                  66           15695.42
3     /api/payments           103                  67           15514.89
4       /api/search           101                  68           14754.39
5        /api/users           100                  67           16232.70
6          /metrics            97                  68           15672.82
7         /api/cart            94                  64           15362.94
8  /api/auth/logout            93                  60           17692.87
9     /api/products            88                  60           15064.74


In [ ]:
## 4. PERFORMANCE POR ENDPOINT
## ¿Qué endpoints son más lentos?

query = """
SELECT
    endpoint,
    COUNT(*) as requests,
    ROUND(AVG(response_time_ms), 2) as avg_time,
    ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY response_time_ms), 2) as p50,
    ROUND(PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY response_time_ms), 2) as p95,
    MAX(response_time_ms) as max_time
FROM access_logs
WHERE status_code < 500  -- Solo requests exitosas
GROUP BY endpoint
HAVING COUNT(*) > 100  -- Solo endpoints con suficiente tráfico
ORDER BY p95 DESC
LIMIT 10;
"""

resultado = con.execute(query).fetchdf()
print(resultado)

           endpoint  requests  avg_time    p50     p95  max_time
0     /api/products       358    262.61  265.5  482.30       500
1     /api/payments       371    261.15  250.0  477.50       500
2        /api/users       377    256.03  257.0  477.40       500
3          /metrics       383    249.10  250.0  476.70       500
4   /api/auth/login       332    263.70  270.5  474.45       500
5  /api/auth/logout       380    241.98  240.0  474.05       499
6     /api/checkout       335    253.40  253.0  472.60       500
7       /api/search       338    238.55  237.0  468.45       499
8           /health       368    241.54  230.5  468.30       499
9       /api/orders       373    241.25  238.0  466.40       500


In [ ]:
## 5. TENDENCIA HORARIA
## ¿A qué hora hay más tráfico?

query = """
SELECT
    EXTRACT(HOUR FROM timestamp) as hora,
    COUNT(*) as requests,
    ROUND(AVG(response_time_ms), 2) as avg_response_time,
    SUM(CASE WHEN status_code >= 500 THEN 1 ELSE 0 END) as errors
FROM access_logs
GROUP BY EXTRACT(HOUR FROM timestamp)
ORDER BY hora;
"""

resultado = con.execute(query).fetchdf()
print(resultado)

    hora  requests  avg_response_time  errors
0      0       213            3769.90    49.0
1      1       180            3054.24    31.0
2      2       197            3351.84    39.0
3      3       184            3911.88    39.0
4      4       204            3733.99    48.0
5      5       225            4161.73    54.0
6      6       222            3767.15    48.0
7      7       210            3515.50    46.0
8      8       215            3215.68    42.0
9      9       194            3329.16    38.0
10    10       207            3704.79    43.0
11    11       213            3435.09    42.0
12    12       221            3241.71    42.0
13    13       226            4088.35    55.0
14    14       211            4227.68    48.0
15    15       191            3236.24    47.0
16    16       216            4100.81    47.0
17    17       204            3599.55    46.0
18    18       212            3344.11    42.0
19    19       203            2640.20    36.0
20    20       210            4072

In [ ]:
## 6. WINDOW FUNCTIONS - RANKING
## Top 3 requests más lentas por endpoint

query = """
WITH ranked AS (
    SELECT
        endpoint,
        timestamp,
        response_time_ms,
        user_id,
        ROW_NUMBER() OVER (
            PARTITION BY endpoint
            ORDER BY response_time_ms DESC
        ) as rank
    FROM access_logs
    WHERE status_code < 500
)
SELECT * FROM ranked
WHERE rank <= 3
ORDER BY endpoint, rank;
"""

resultado = con.execute(query).fetchdf()
print(resultado)

            endpoint           timestamp  response_time_ms  user_id  rank
0    /api/auth/login 2024-09-30 18:32:51               500     <NA>     1
1    /api/auth/login 2026-02-08 02:43:36               500     8485     2
2    /api/auth/login 2024-01-31 01:25:05               500     2836     3
3   /api/auth/logout 2025-03-10 10:07:57               499     8423     1
4   /api/auth/logout 2024-09-26 14:56:07               499     <NA>     2
5   /api/auth/logout 2024-10-06 06:10:59               499     <NA>     3
6          /api/cart 2025-03-14 13:14:28               499       34     1
7          /api/cart 2026-02-07 08:39:21               498      143     2
8          /api/cart 2025-12-18 00:49:48               494     8060     3
9      /api/checkout 2026-03-15 15:14:45               500     <NA>     1
10     /api/checkout 2024-11-11 05:58:04               499     <NA>     2
11     /api/checkout 2024-01-27 06:41:05               498     5231     3
12       /api/orders 2024-11-01 13:09:

In [ ]:
## 7. COMPARACIÓN CON PERÍODO ANTERIOR
## ¿Cómo cambia el tráfico día a día?

query = """
WITH daily_stats AS (
    SELECT
        DATE(timestamp) as fecha,
        COUNT(*) as requests,
        ROUND(AVG(response_time_ms), 2) as avg_time
    FROM access_logs
    GROUP BY DATE(timestamp)
)
SELECT
    fecha,
    requests,
    LAG(requests) OVER (ORDER BY fecha) as requests_dia_anterior,
    requests - LAG(requests) OVER (ORDER BY fecha) as diferencia,
    ROUND(
        (requests - LAG(requests) OVER (ORDER BY fecha)) * 100.0 /
        LAG(requests) OVER (ORDER BY fecha),
        2
    ) as cambio_porcentual
FROM daily_stats
ORDER BY fecha;
"""

resultado = con.execute(query).fetchdf()
print(resultado)

         fecha  requests  requests_dia_anterior  diferencia  cambio_porcentual
0   2024-01-01         5                   <NA>        <NA>                NaN
1   2024-01-02         3                      5          -2             -40.00
2   2024-01-03         3                      3           0               0.00
3   2024-01-04         9                      3           6             200.00
4   2024-01-05         7                      9          -2             -22.22
..         ...       ...                    ...         ...                ...
819 2026-04-04         4                      5          -1             -20.00
820 2026-04-05         6                      4           2              50.00
821 2026-04-06         8                      6           2              33.33
822 2026-04-07         5                      8          -3             -37.50
823 2026-04-08         6                      5           1              20.00

[824 rows x 5 columns]


## Based on your analysis, write 3 actionable recommendations for dev team.

The /api/search endpoint shows a p95 response time of ~2.5 seconds, which is significantly higher than other endpoints.

I recommend implementing caching (e.g., Redis) for frequent queries and reviewing the underlying database queries for optimization (indexes, query plans).

The /api/login endpoint represents ~35% of total requests, making it the most heavily used endpoint.

I recommend optimizing this endpoint for scalability by adding rate limiting, improving authentication performance, and ensuring it is horizontally scalable.

We observed X unique users generating a high volume of requests within a short time window, indicating traffic spikes or potential load issues.

I recommend implementing load balancing and autoscaling strategies, as well as monitoring peak traffic periods to prevent performance degradation.